In [1]:
"""
=============================================================
  Skin Lesion Classification - Fill All Three Tables
  Dataset: nodoubttome/skin-cancer9-classesisic (Kaggle)

  Folder structure after download:
    <path>/
      Skin cancer ISIC The International Skin Imaging Collaboration/
        Train/
          melanoma/
          nevus/
          ...
        Test/
          melanoma/
          nevus/
          ...

  Tables:
    1. Transfer Learning Models Comparison
    2. Deep Features + Classical Classifiers
    3. Computational Efficiency Comparison
=============================================================
"""

import os, time, copy, json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torchvision import transforms, models, datasets

import timm

from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

try:
    from thop import profile as thop_profile
    THOP_AVAILABLE = True
except ImportError:
    THOP_AVAILABLE = False

# ─────────────────────────────────────────────
#  STEP 1 – DOWNLOAD DATASET VIA KAGGLEHUB
# ─────────────────────────────────────────────
import kagglehub
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Dataset path:", path)

# Auto-detect Train/Test folders
BASE = None
for root, dirs, files in os.walk(path):
    if 'Train' in dirs and 'Test' in dirs:
        BASE = root
        break

if BASE is None:
    # Try direct path
    if os.path.exists(os.path.join(path, 'Train')):
        BASE = path
    else:
        # Search one level deeper
        for d in os.listdir(path):
            candidate = os.path.join(path, d)
            if os.path.isdir(candidate):
                if os.path.exists(os.path.join(candidate, 'Train')):
                    BASE = candidate
                    break

print(f"Base directory: {BASE}")
TRAIN_DIR = os.path.join(BASE, 'Train')
TEST_DIR  = os.path.join(BASE, 'Test')

# Check classes — restrict to only 5 of the 9 available
SELECTED_CLASSES = ['melanoma', 'basal cell carcinoma', 'nevus',
                     'vascular lesion', 'dermatofibroma']  # change this list to pick different classes

available_classes = sorted(os.listdir(TRAIN_DIR))
missing = [c for c in SELECTED_CLASSES if c not in available_classes]
if missing:
    raise ValueError(f"Classes not found: {missing}. Available: {available_classes}")

classes = sorted(SELECTED_CLASSES)
NUM_CLASSES = len(classes)
print(f"Using {NUM_CLASSES} of {len(available_classes)} available classes: {classes}")

SAVE_DIR = "./results"
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE   = 224
BATCH_SIZE = 16
EPOCHS     = 10
LR         = 1e-4
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ─────────────────────────────────────────────
#  TRANSFORMS
# ─────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ─────────────────────────────────────────────
#  RESTRICT AN ImageFolder DATASET TO SELECTED CLASSES
# ─────────────────────────────────────────────
def filter_imagefolder_by_classes(ds, selected_classes):
    old_class_to_idx = ds.class_to_idx
    keep_old_idx = {old_class_to_idx[c] for c in selected_classes}
    new_classes = sorted(selected_classes)
    old_to_new = {old_class_to_idx[c]: i for i, c in enumerate(new_classes)}

    new_samples = [(path, old_to_new[label]) for path, label in ds.samples if label in keep_old_idx]
    ds.samples = new_samples
    ds.imgs = new_samples
    ds.targets = [label for _, label in new_samples]
    ds.classes = new_classes
    ds.class_to_idx = {c: i for i, c in enumerate(new_classes)}
    return ds


# ─────────────────────────────────────────────
#  DATA LOADERS
# ─────────────────────────────────────────────
def make_loaders():
    full_train = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
    test_ds    = datasets.ImageFolder(TEST_DIR,  transform=val_tf)

    full_train = filter_imagefolder_by_classes(full_train, SELECTED_CLASSES)
    test_ds    = filter_imagefolder_by_classes(test_ds, SELECTED_CLASSES)


    # 90% train / 10% val split from Train folder
    n_val   = int(0.1 * len(full_train))
    n_train = len(full_train) - n_val
    train_ds, val_ds = random_split(
        full_train, [n_train, n_val],
        generator=torch.Generator().manual_seed(42))

    # Apply val transform to val split
    val_ds.dataset.transform = val_tf

    # Weighted sampler for class imbalance
    labels      = [full_train.targets[i] for i in train_ds.indices]
    class_counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    weights      = 1.0 / class_counts[labels]
    sampler      = WeightedRandomSampler(weights, len(weights))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=2, pin_memory=True)

    print(f"\nTrain: {n_train} | Val: {n_val} | Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader, full_train.class_to_idx

# ─────────────────────────────────────────────
#  MODEL FACTORY
# ─────────────────────────────────────────────
def get_model(name):
    n = name.lower().replace('-','').replace('_','')

    if n == 'alexnet':
        m = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)

    elif n == 'vgg16':
        m = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)

    elif n == 'vgg19':
        m = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)

    elif n == 'resnet18':
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)

    elif n == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)

    elif n == 'resnet101':
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)

    elif n == 'densenet121':
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        m.classifier = nn.Linear(m.classifier.in_features, NUM_CLASSES)

    elif n == 'efficientnetb0':
        m = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)

    else:
        raise ValueError(f"Unknown model: {name}")

    return m

# ─────────────────────────────────────────────
#  TRAIN ONE EPOCH
# ─────────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss/total, correct/total

# ─────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            out  = model(imgs)
            probs = torch.softmax(out, dim=1).cpu().numpy()
            preds = out.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs)

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    acc  = accuracy_score(all_labels, all_preds) * 100
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0) * 100
    rec  = recall_score(all_labels, all_preds,    average='weighted', zero_division=0) * 100
    f1   = f1_score(all_labels, all_preds,         average='weighted', zero_division=0) * 100
    try:
        from sklearn.preprocessing import label_binarize
        y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
        auc = roc_auc_score(y_bin, all_probs,
                            multi_class='ovr', average='weighted') * 100
    except Exception:
        auc = 0.0

    return round(acc,2), round(prec,2), round(rec,2), round(f1,2), round(auc,2)

# ─────────────────────────────────────────────
#  EFFICIENCY METRICS  (Table 3)
# ─────────────────────────────────────────────
def compute_efficiency(model):
    model.eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

    params_M = sum(p.numel() for p in model.parameters()) / 1e6
    size_MB  = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**2)

    flops_G = 'N/A'
    if THOP_AVAILABLE:
        try:
            flops, _ = thop_profile(model, inputs=(dummy,), verbose=False)
            flops_G  = round(flops / 1e9, 2)
        except:
            pass

    # Inference time – avg over 50 runs
    with torch.no_grad():
        for _ in range(5): model(dummy)   # warmup
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(50): model(dummy)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        inf_ms = (time.perf_counter() - t0) / 50 * 1000

    return round(params_M,2), round(size_MB,2), flops_G, round(inf_ms,2)

# ─────────────────────────────────────────────
#  TABLE 1 + TABLE 3
# ─────────────────────────────────────────────
def run_table1_and_3(train_loader, val_loader, test_loader):
    model_names = ['AlexNet','VGG16','VGG19',
                   'ResNet18','ResNet50','ResNet101',
                   'DenseNet121','EfficientNet-B0']

    table1, table3 = {}, {}

    for mname in model_names:
        print(f"\n{'='*60}")
        print(f"  Training: {mname}")
        print(f"{'='*60}")

        model     = get_model(mname).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LR)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

        best_val, best_w = 0, None
        for ep in range(1, EPOCHS+1):
            tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
            val_acc, *_     = evaluate(model, val_loader)
            scheduler.step()
            print(f"  Ep {ep:02d}/{EPOCHS} | Loss:{tr_loss:.4f} | "
                  f"TrainAcc:{tr_acc*100:.1f}% | ValAcc:{val_acc:.1f}%")
            if val_acc > best_val:
                best_val = val_acc
                best_w   = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_w)
        acc, prec, rec, f1, auc = evaluate(model, test_loader)
        print(f"\n  ✔ TEST → Acc:{acc}  Prec:{prec}  Rec:{rec}  F1:{f1}  AUC:{auc}")

        table1[mname] = {'Accuracy (%)':acc,'Precision (%)':prec,
                         'Recall (%)':rec,'F1-Score (%)':f1,'AUC (%)':auc}

        p, s, fl, im = compute_efficiency(model)
        table3[mname] = {'Parameters (M)':p,'Model Size (MB)':s,
                         'FLOPs (G)':fl,'Inference Time (ms)':im,'Accuracy (%)':acc}

        ckpt = os.path.join(SAVE_DIR, mname.replace('-','_')+'_best.pth')
        torch.save(model.state_dict(), ckpt)

        del model
        if DEVICE.type=='cuda': torch.cuda.empty_cache()

    return table1, table3

# ─────────────────────────────────────────────
#  FEATURE EXTRACTION
# ─────────────────────────────────────────────
def extract_features(backbone_name, train_loader, test_loader):
    ckpt = os.path.join(SAVE_DIR, backbone_name.replace('-','_')+'_best.pth')
    if not os.path.exists(ckpt):
        print(f"  No checkpoint for {backbone_name}. Run Table 1 first.")
        return None, None, None, None

    model = get_model(backbone_name)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model = model.to(DEVICE).eval()

    captured = []
    def hook(module, inp, out):
        captured.append(out.detach().cpu())

    n = backbone_name.lower().replace('-','').replace('_','')
    if 'alexnet' in n or 'vgg' in n:
        h = model.classifier[-2].register_forward_hook(hook)
    elif 'resnet' in n:
        h = model.avgpool.register_forward_hook(hook)
    elif 'densenet' in n:
        h = model.features.register_forward_hook(hook)
    elif 'efficientnet' in n:
        h = model.global_pool.register_forward_hook(hook)
    else:
        h = list(model.children())[-2].register_forward_hook(hook)

    def collect(loader):
        feats, labs = [], []
        with torch.no_grad():
            for imgs, labels in loader:
                captured.clear()
                model(imgs.to(DEVICE))
                f = captured[0].view(captured[0].size(0), -1)
                feats.append(f.numpy())
                labs.extend(labels.numpy())
        return np.vstack(feats), np.array(labs)

    X_tr, y_tr = collect(train_loader)
    X_te, y_te = collect(test_loader)
    h.remove()
    del model
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return X_tr, y_tr, X_te, y_te

# ─────────────────────────────────────────────
#  TABLE 2
# ─────────────────────────────────────────────
def run_table2(train_loader, test_loader, best_backbone='ResNet50'):
    print(f"\n{'='*60}")
    print(f"  TABLE 2 – Features from: {best_backbone}")
    print(f"{'='*60}")

    X_tr, y_tr, X_te, y_te = extract_features(best_backbone, train_loader, test_loader)
    if X_tr is None:
        return {}

    classifiers = {
        'Logistic Regression': LogisticRegression(max_iter=500, solver='lbfgs',
                                                   multi_class='auto', n_jobs=-1),
        'Decision Tree':       DecisionTreeClassifier(max_depth=15, random_state=42),
        'Random Forest':       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'Linear SVM':          SVC(kernel='linear', probability=True, max_iter=2000),
        'RBF-SVM':             SVC(kernel='rbf', probability=True, C=10, gamma='scale'),
        'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.1,
                                             eval_metric='mlogloss', n_jobs=-1, random_state=42),
    }

    results = {}
    for clf_name, clf in classifiers.items():
        print(f"  Fitting {clf_name}...", end=' ', flush=True)
        t0 = time.time()
        clf.fit(X_tr, y_tr)
        print(f"{time.time()-t0:.1f}s")

        preds = clf.predict(X_te)
        acc   = accuracy_score(y_te, preds) * 100
        prec  = precision_score(y_te, preds, average='weighted', zero_division=0) * 100
        rec   = recall_score(y_te, preds,    average='weighted', zero_division=0) * 100
        f1    = f1_score(y_te, preds,         average='weighted', zero_division=0) * 100

        try:
            if hasattr(clf, 'predict_proba'):
                prob = clf.predict_proba(X_te)
            else:
                prob = np.eye(NUM_CLASSES)[preds]
            from sklearn.preprocessing import label_binarize
            y_bin = label_binarize(y_te, classes=list(range(NUM_CLASSES)))
            auc = roc_auc_score(y_bin, prob, multi_class='ovr', average='weighted') * 100
        except:
            auc = 0.0

        r = {k: round(v,2) for k,v in
             zip(['Accuracy (%)','Precision (%)','Recall (%)','F1-Score (%)','AUC (%)'],
                 [acc, prec, rec, f1, auc])}
        r['Feature Extractor'] = best_backbone
        r['Classifier']        = clf_name
        results[clf_name] = r
        print(f"    → Acc:{r['Accuracy (%)']}  Prec:{r['Precision (%)']}  "
              f"Rec:{r['Recall (%)']}  F1:{r['F1-Score (%)']}  AUC:{r['AUC (%)']}")

    return results

# ─────────────────────────────────────────────
#  PRINT TABLES
# ─────────────────────────────────────────────
def print_all(t1, t2, t3):
    SEP = "─" * 72

    print(f"\n\n{'═'*72}")
    print("  TABLE 1 – Transfer Learning Model Comparison")
    print(f"{'═'*72}")
    print(f"{'Model':<18} {'Acc%':>7} {'Prec%':>7} {'Rec%':>7} {'F1%':>7} {'AUC%':>7}")
    print(SEP)
    for m, r in t1.items():
        print(f"{m:<18} {r['Accuracy (%)']:>7} {r['Precision (%)']:>7} "
              f"{r['Recall (%)']:>7} {r['F1-Score (%)']:>7} {r['AUC (%)']:>7}")

    print(f"\n\n{'═'*72}")
    print("  TABLE 2 – Deep Features + Classical Classifiers")
    print(f"{'═'*72}")
    print(f"{'Classifier':<25} {'Acc%':>7} {'Prec%':>7} {'Rec%':>7} {'F1%':>7} {'AUC%':>7}")
    print(SEP)
    for c, r in t2.items():
        print(f"{c:<25} {r['Accuracy (%)']:>7} {r['Precision (%)']:>7} "
              f"{r['Recall (%)']:>7} {r['F1-Score (%)']:>7} {r['AUC (%)']:>7}")

    print(f"\n\n{'═'*72}")
    print("  TABLE 3 – Computational Efficiency")
    print(f"{'═'*72}")
    print(f"{'Model':<18} {'Params(M)':>10} {'Size(MB)':>10} "
          f"{'FLOPs(G)':>10} {'Infer(ms)':>10} {'Acc%':>7}")
    print(SEP)
    for m, r in t3.items():
        print(f"{m:<18} {r['Parameters (M)']:>10} {r['Model Size (MB)']:>10} "
              f"{str(r['FLOPs (G)']):>10} {r['Inference Time (ms)']:>10} "
              f"{r['Accuracy (%)']:>7}")

# ─────────────────────────────────────────────
#  SAVE
# ─────────────────────────────────────────────
def save_all(t1, t2, t3):
    with open(os.path.join(SAVE_DIR,'all_results.json'),'w') as f:
        json.dump({'table1':t1,'table2':t2,'table3':t3}, f, indent=2)
    pd.DataFrame(t1).T.to_csv(os.path.join(SAVE_DIR,'table1.csv'))
    pd.DataFrame(t2).T.to_csv(os.path.join(SAVE_DIR,'table2.csv'))
    pd.DataFrame(t3).T.to_csv(os.path.join(SAVE_DIR,'table3.csv'))
    print(f"\n✔ Saved to {SAVE_DIR}/ (JSON + 3 CSV files)")

# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────
if __name__ == '__main__':
    # ── Install missing packages if needed ──
    import subprocess, sys
    for pkg in ['timm','xgboost','thop']:
        try: __import__(pkg)
        except ImportError:
            subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

    # ── Loaders ──────────────────────────────
    train_loader, val_loader, test_loader, class_map = make_loaders()
    print("Class mapping:", class_map)

    # ── Table 1 + 3 ──────────────────────────
    table1, table3 = run_table1_and_3(train_loader, val_loader, test_loader)

    # ── Pick best backbone for Table 2 ───────
    best_backbone = max(table1, key=lambda m: table1[m]['Accuracy (%)'])
    print(f"\nBest backbone for feature extraction: {best_backbone} "
          f"({table1[best_backbone]['Accuracy (%)']}%)")

    # ── Table 2 ──────────────────────────────
    table2 = run_table2(train_loader, test_loader, best_backbone)

    # ── Print + Save ──────────────────────────
    print_all(table1, table2, table3)
    save_all(table1, table2, table3)

    print("\n\n✅ ALL DONE! Copy numbers from ./results/*.csv into your .docx tables.")

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Dataset path: /kaggle/input/skin-cancer9-classesisic
Base directory: /kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration
Using 5 of 9 available classes: ['basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'vascular lesion']
Device: cuda

Train: 1265 | Val: 140 | Test: 67
Class mapping: {'basal cell carcinoma': 0, 'dermatofibroma': 1, 'melanoma': 2, 'nevus': 3, 'vascular lesion': 4}

  Training: AlexNet
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 175MB/s]


  Ep 01/10 | Loss:0.8194 | TrainAcc:67.6% | ValAcc:71.4%
  Ep 02/10 | Loss:0.4174 | TrainAcc:84.3% | ValAcc:80.7%
  Ep 03/10 | Loss:0.2983 | TrainAcc:89.6% | ValAcc:84.3%
  Ep 04/10 | Loss:0.2122 | TrainAcc:92.3% | ValAcc:79.3%
  Ep 05/10 | Loss:0.1357 | TrainAcc:95.0% | ValAcc:87.9%
  Ep 06/10 | Loss:0.0782 | TrainAcc:97.6% | ValAcc:90.0%
  Ep 07/10 | Loss:0.0376 | TrainAcc:98.7% | ValAcc:87.9%
  Ep 08/10 | Loss:0.0346 | TrainAcc:99.1% | ValAcc:87.1%
  Ep 09/10 | Loss:0.0342 | TrainAcc:99.1% | ValAcc:89.3%
  Ep 10/10 | Loss:0.0316 | TrainAcc:99.0% | ValAcc:88.6%

  ✔ TEST → Acc:73.13  Prec:78.15  Rec:73.13  F1:72.19  AUC:89.14

  Training: VGG16
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:10<00:00, 50.8MB/s]


  Ep 01/10 | Loss:1.0123 | TrainAcc:60.9% | ValAcc:56.4%
  Ep 02/10 | Loss:0.5211 | TrainAcc:82.1% | ValAcc:77.9%
  Ep 03/10 | Loss:0.3730 | TrainAcc:87.2% | ValAcc:76.4%
  Ep 04/10 | Loss:0.3245 | TrainAcc:87.7% | ValAcc:74.3%
  Ep 05/10 | Loss:0.2201 | TrainAcc:91.9% | ValAcc:86.4%
  Ep 06/10 | Loss:0.1218 | TrainAcc:95.5% | ValAcc:85.0%
  Ep 07/10 | Loss:0.1017 | TrainAcc:96.5% | ValAcc:85.7%
  Ep 08/10 | Loss:0.0456 | TrainAcc:98.8% | ValAcc:90.0%
  Ep 09/10 | Loss:0.0227 | TrainAcc:98.9% | ValAcc:87.1%
  Ep 10/10 | Loss:0.0404 | TrainAcc:98.7% | ValAcc:87.1%

  ✔ TEST → Acc:59.7  Prec:56.87  Rec:59.7  F1:53.1  AUC:84.66

  Training: VGG19
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

100%|██████████| 30.8M/30.8M [00:00<00:00, 189MB/s]


  Ep 01/10 | Loss:0.7560 | TrainAcc:76.3% | ValAcc:80.0%
  Ep 02/10 | Loss:0.3050 | TrainAcc:90.5% | ValAcc:80.7%
  Ep 03/10 | Loss:0.1941 | TrainAcc:93.8% | ValAcc:82.9%
  Ep 04/10 | Loss:0.1427 | TrainAcc:96.3% | ValAcc:83.6%
  Ep 05/10 | Loss:0.1060 | TrainAcc:97.2% | ValAcc:85.0%
  Ep 06/10 | Loss:0.0717 | TrainAcc:98.3% | ValAcc:87.1%
  Ep 07/10 | Loss:0.0592 | TrainAcc:98.6% | ValAcc:84.3%
  Ep 08/10 | Loss:0.0447 | TrainAcc:99.4% | ValAcc:89.3%
  Ep 09/10 | Loss:0.0426 | TrainAcc:99.3% | ValAcc:87.9%
  Ep 10/10 | Loss:0.0326 | TrainAcc:99.4% | ValAcc:86.4%

  ✔ TEST → Acc:70.15  Prec:81.07  Rec:70.15  F1:68.37  AUC:92.65

  Training: EfficientNet-B0


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

  Ep 01/10 | Loss:1.3542 | TrainAcc:60.9% | ValAcc:56.4%
  Ep 02/10 | Loss:0.4597 | TrainAcc:84.3% | ValAcc:69.3%
  Ep 03/10 | Loss:0.2899 | TrainAcc:91.1% | ValAcc:62.1%
  Ep 04/10 | Loss:0.2007 | TrainAcc:92.6% | ValAcc:70.0%
  Ep 05/10 | Loss:0.1426 | TrainAcc:95.4% | ValAcc:72.1%
  Ep 06/10 | Loss:0.0971 | TrainAcc:96.4% | ValAcc:73.6%
  Ep 07/10 | Loss:0.0867 | TrainAcc:97.0% | ValAcc:75.0%
  Ep 08/10 | Loss:0.0968 | TrainAcc:96.8% | ValAcc:77.1%
  Ep 09/10 | Loss:0.0515 | TrainAcc:98.7% | ValAcc:75.0%
  Ep 10/10 | Loss:0.0387 | TrainAcc:98.7% | ValAcc:73.6%

  ✔ TEST → Acc:56.72  Prec:56.98  Rec:56.72  F1:51.2  AUC:86.98

Best backbone for feature extraction: AlexNet (73.13%)

  TABLE 2 – Features from: AlexNet
  Fitting Logistic Regression... 2.1s
    → Acc:65.67  Prec:69.84  Rec:65.67  F1:62.71  AUC:83.76
  Fitting Decision Tree... 0.7s
    → Acc:67.16  Prec:68.46  Rec:67.16  F1:65.28  AUC:78.43
  Fitting Random Forest... 1.7s
    → Acc:67.16  Prec:71.91  Rec:67.16  F1:64.37  A